# Enterprise Track 3 · Governance: Who Sees What

**Goal:** apply the course's rule for sensitive attributes on the platform itself. Race and religion are collected for required reporting and accommodations; access to them is restricted. You'll use the three tools Unity Catalog gives you: **tags** (what a column is), **column masks and row filters** (what each reader sees), and **views with grants** (what each reader can query at all).

> ⚠️ **Mock data only.** This track runs on the agency workspace, which is approved for criminal justice information, but every table here holds the course's fictional records. Real tables come later, with the data owner's approval (see `enterprise/README.md`).

In [ ]:
# ▶ SETUP: run this cell first (Shift + Enter). It finds the course files and checks your settings.
STEP = 3

import importlib
import sys
from pathlib import Path

def _find_course_files():
    for folder in [Path.cwd(), *Path.cwd().parents]:          # the course's Git folder in Databricks
        if (folder / "lnl" / "__init__.py").exists():
            return folder
    try:                                                         # found from the notebook's own location
        here = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        for folder in Path("/Workspace" + here).parents:
            if (folder / "lnl" / "__init__.py").exists():
                return folder
    except Exception:
        pass
    raise RuntimeError("Can't find the course files. Open this notebook from inside the course's Git folder.")

COURSE = _find_course_files()
sys.path.insert(0, str(COURSE))
from lnl import config, enterprise as ent
importlib.reload(config)                                         # picks up edits to lnl/config.py without a restart
N = ent.prepare(STEP, "Governance")

## 1. Three tools, three jobs
| Tool | What it does | Follows the data? |
|---|---|---|
| **Tags** | label a table or column (`sensitivity = restricted`) so people and policies can find it | yes, it's on the object |
| **Column mask / row filter** | a SQL function decides, per reader, what each value or row shows | yes: every query of the table, from any tool, is filtered |
| **View + grants** | publish a curated shape; readers get the view, not the table | only for people who read through the view |

Masks and filters are the stronger control because they live on the table. On a table a pipeline manages, they're declared in the pipeline's definition (the `schema=` and `row_filter=` options of `@dp.table`; row filters there are in Public Preview), because each update re-runs the definition and can undo an `ALTER`. To practice the syntax, this notebook applies them to a small regular table cut from silver.

The deciding function is `is_account_group_member('<group>')`. Your settings: `RESTRICTED_GROUP` decides who sees race and religion; if it isn't set, a group nobody is in is used, so everything stays masked. That's the safe default.

In [ ]:
restricted = N["restricted_group"]
print(f"Restricted group: {restricted}")
print("You are", "IN" if spark.sql(f"SELECT is_account_group_member('{restricted}')").first()[0] else "NOT in", "it")

## 2. A practice table and tags
Two hundred records from four units. Tags record *what* each sensitive column is; `information_schema.column_tags` makes that searchable across every catalog you can see.

In [ ]:
ent.sql(f"""
    CREATE OR REPLACE TABLE {N['fq']}.mask_demo
    COMMENT 'Practice table for masks and row filters (mock data)'
    AS SELECT tdcj_number, unit_code, unit_name, race_code, religion_group, stg_status
    FROM {N['fq']}.silver_offenders
    WHERE unit_code IN ('GOC', 'MET', 'CEC', 'STC')
    LIMIT 200""")
for column in ["race_code", "religion_group"]:
    ent.set_tags(f"ALTER TABLE {N['fq']}.mask_demo ALTER COLUMN {column} SET TAGS ('sensitivity' = 'restricted', 'reason' = 'protected attribute')")
ent.set_tags(f"ALTER TABLE {N['fq']}.mask_demo ALTER COLUMN stg_status SET TAGS ('sensitivity' = 'law_enforcement')")
display(ent.sql(f"""
    SELECT table_name, column_name, tag_name, tag_value
    FROM {N['catalog']}.information_schema.column_tags
    WHERE schema_name = '{N['schema']}' ORDER BY column_name, tag_name"""))

## 3. A column mask
A mask is a SQL function that receives the real value and returns what this reader may see. It runs on every query, from any tool, including yours as the owner.

In [ ]:
ent.sql(f"""
    CREATE OR REPLACE FUNCTION {N['fq']}.mask_restricted(value STRING)
    RETURN CASE WHEN is_account_group_member('{restricted}') THEN value ELSE 'RESTRICTED' END""")
for column in ["race_code", "religion_group"]:
    ent.sql(f"ALTER TABLE {N['fq']}.mask_demo ALTER COLUMN {column} SET MASK {N['fq']}.mask_restricted")
display(spark.table(f"{N['fq']}.mask_demo").limit(8))

## 4. A row filter with a mapping table
Row filters decide which **rows** a reader sees. A common pattern is a small mapping table that says who may see which units, so access is data you can review rather than code. Here you give yourself two of the four units.

In [ ]:
ent.sql(f"CREATE TABLE IF NOT EXISTS {N['fq']}.unit_access (user_name STRING, unit_code STRING) COMMENT 'Who may see which units'")
ent.sql(f"DELETE FROM {N['fq']}.unit_access WHERE user_name = current_user()")
ent.sql(f"INSERT INTO {N['fq']}.unit_access VALUES (current_user(), 'GOC'), (current_user(), 'MET')")
ent.sql(f"""
    CREATE OR REPLACE FUNCTION {N['fq']}.only_my_units(u STRING)
    RETURN EXISTS (SELECT 1 FROM {N['fq']}.unit_access a WHERE a.unit_code = u AND a.user_name = current_user())""")
ent.sql(f"ALTER TABLE {N['fq']}.mask_demo SET ROW FILTER {N['fq']}.only_my_units ON (unit_code)")
display(ent.sql(f"SELECT unit_name, unit_code, race_code, count(*) AS records FROM {N['fq']}.mask_demo GROUP BY ALL ORDER BY 1"))

Four units went in; you see two, and race is masked in both. Add `'CEC'` to `unit_access` and run the query again: the third unit appears, no code changed. Remove the filter or masks with `ALTER TABLE ... DROP ROW FILTER` and `ALTER TABLE ... ALTER COLUMN ... DROP MASK`.

## 5. The published view
Analysts shouldn't query silver directly: it carries technical columns, and every new column would be exposed by default. A **view** publishes the agreed shape. This one masks race and religion with the same group check, and analysts get `SELECT` on the view only. The view runs with its owner's (your) access to silver, so analysts never need access to silver itself.

In [ ]:
ent.sql(f"""
    CREATE OR REPLACE VIEW {N['fq']}.offenders_analyst
    COMMENT 'Published view of the cleansed mock records. Race and religion are masked outside {restricted}.'
    AS SELECT
      tdcj_number, sid_number, last_name_std AS last_name, first_name_std AS first_name, alias,
      unit_name, unit_code, custody_level, threat_level, violence_code_std AS violence_code,
      stg_group, stg_status, offense_code_std AS offense_code, offense_against,
      sentence_date, projected_release_date, age_at_sentencing,
      CASE WHEN is_account_group_member('{restricted}') THEN race_code ELSE 'RESTRICTED' END AS race_code,
      CASE WHEN is_account_group_member('{restricted}') THEN race_omb ELSE 'RESTRICTED' END AS race_omb,
      CASE WHEN is_account_group_member('{restricted}') THEN religion_group ELSE 'RESTRICTED' END AS religion_group
    FROM {N['fq']}.silver_offenders""")
group = config.ANALYST_GROUP
statement = f"GRANT SELECT ON TABLE {N['fq']}.offenders_analyst TO `{group or '<analyst group>'}`"
ent.sql(statement) if group else print(statement + ";   -- not run: set ANALYST_GROUP in lnl/config.py")
display(spark.table(f"{N['fq']}.offenders_analyst").limit(5))

## 6. Where this goes at scale: governed tags and ABAC
Masking table by table doesn't scale to hundreds of tables. **Attribute-based access control (ABAC)** turns it around: an account admin defines **governed tags** (say `sensitivity`, with a fixed list of allowed values and control over who may assign them), data stewards tag columns, and one **policy** on a catalog or schema masks every column carrying the tag, in tables that exist now and tables created later. The tags you set above are ordinary tags: they label and search the same way, but a policy only reads governed ones. Governed tags are account-admin work; a policy can be created by anyone with `MANAGE` on the catalog or schema (or its owner). Ask what your workspace has enabled.

## 7. Lineage
Open **Catalog**, find `offenders_analyst`, and open **Lineage**: Unity Catalog recorded that it reads `silver_offenders`, which reads `bronze_offenders`, which reads the landing volume. Nobody documented that by hand. Notebook 05 queries the same lineage with SQL.

**Next:** 04_job_and_alerts runs the pipeline on a schedule, checks quality after every run, and pages someone when a check fails.